Setup dan Koneksi (Persiapan)

In [8]:
import pymongo
from pyspark.sql import SparkSession
import pandas as pd

# Koneksi ke MongoDB lokal
client = pymongo.MongoClient("mongodb://localhost:27017/")
db = client["kopi_nusantara"]
koleksi_menu = db["menu"]

# Hapus koleksi jika sudah ada agar data tidak duplikat saat di-run ulang
koleksi_menu.drop()

A. Membangun Katalog (Create)

In [9]:
# Menyisipkan minimal 12 dokumen dengan variasi field[cite: 9]
data_menu = [
    # Kategori Kopi (wajib ada tingkat_kepahitan)[cite: 9]
    {"nama_menu": "Espresso", "harga": 20000, "kategori": "Kopi", "tingkat_kepahitan": 5, "stok": 50},
    {"nama_menu": "Americano", "harga": 22000, "kategori": "Kopi", "tingkat_kepahitan": 4},
    {"nama_menu": "Cappuccino", "harga": 28000, "kategori": "Kopi", "tingkat_kepahitan": 2},
    {"nama_menu": "Latte", "harga": 30000, "kategori": "Kopi", "tingkat_kepahitan": 2, "rekomendasi_barista": True},
    {"nama_menu": "Mocha", "harga": 32000, "kategori": "Kopi", "tingkat_kepahitan": 3},
    {"nama_menu": "Macchiato", "harga": 26000, "kategori": "Kopi", "tingkat_kepahitan": 4},
    
    # Kategori Makanan (wajib ada pedas_level)[cite: 9]
    {"nama_menu": "Nasi Goreng Spesial", "harga": 35000, "kategori": "Makanan", "pedas_level": 2},
    {"nama_menu": "Mie Goreng Jawa", "harga": 30000, "kategori": "Makanan", "pedas_level": 1},
    {"nama_menu": "Ayam Geprek", "harga": 25000, "kategori": "Makanan", "pedas_level": 3},
    {"nama_menu": "Kentang Goreng", "harga": 15000, "kategori": "Makanan", "pedas_level": 0},
    {"nama_menu": "Roti Bakar Coklat", "harga": 18000, "kategori": "Makanan", "pedas_level": 0},
    {"nama_menu": "Pisang Goreng Keju", "harga": 16000, "kategori": "Makanan", "pedas_level": 0}
]

koleksi_menu.insert_many(data_menu)
print("Berhasil insert 12 dokumen katalog menu.")

Berhasil insert 12 dokumen katalog menu.


B. Kueri Bertingkat (Read)

In [10]:
print("1. Kopi dengan tingkat kepahitan di atas 3:")
kopi_pahit = koleksi_menu.find({"kategori": "Kopi", "tingkat_kepahitan": {"$gt": 3}})
for menu in kopi_pahit: print(menu)

print("\n2. Menu dengan harga di antara 15.000 dan 30.000:")
harga_menengah = koleksi_menu.find({"harga": {"$gte": 15000, "$lte": 30000}})
for menu in harga_menengah: print(menu)

print("\n3. Menu yang tidak memiliki field pedas_level:")
tanpa_pedas = koleksi_menu.find({"pedas_level": {"$exists": False}})
for menu in tanpa_pedas: print(menu)

1. Kopi dengan tingkat kepahitan di atas 3:
{'_id': ObjectId('6aeb18d478491f997ebb91d1'), 'nama_menu': 'Espresso', 'harga': 20000, 'kategori': 'Kopi', 'tingkat_kepahitan': 5, 'stok': 50}
{'_id': ObjectId('6aeb18d478491f997ebb91d2'), 'nama_menu': 'Americano', 'harga': 22000, 'kategori': 'Kopi', 'tingkat_kepahitan': 4}
{'_id': ObjectId('6aeb18d478491f997ebb91d6'), 'nama_menu': 'Macchiato', 'harga': 26000, 'kategori': 'Kopi', 'tingkat_kepahitan': 4}

2. Menu dengan harga di antara 15.000 dan 30.000:
{'_id': ObjectId('6aeb18d478491f997ebb91d1'), 'nama_menu': 'Espresso', 'harga': 20000, 'kategori': 'Kopi', 'tingkat_kepahitan': 5, 'stok': 50}
{'_id': ObjectId('6aeb18d478491f997ebb91d2'), 'nama_menu': 'Americano', 'harga': 22000, 'kategori': 'Kopi', 'tingkat_kepahitan': 4}
{'_id': ObjectId('6aeb18d478491f997ebb91d3'), 'nama_menu': 'Cappuccino', 'harga': 28000, 'kategori': 'Kopi', 'tingkat_kepahitan': 2}
{'_id': ObjectId('6aeb18d478491f997ebb91d4'), 'nama_menu': 'Latte', 'harga': 30000, 'kateg

C. Pembaruan Data (Update)

In [11]:
# Tambahkan field status "Tersedia" ke seluruh dokumen[cite: 9]
koleksi_menu.update_many({}, {"$set": {"status": "Tersedia"}})

# Ubah status menjadi "Habis" khusus untuk 2 menu pilihan[cite: 9]
koleksi_menu.update_one({"nama_menu": "Espresso"}, {"$set": {"status": "Habis"}})
koleksi_menu.update_one({"nama_menu": "Ayam Geprek"}, {"$set": {"status": "Habis"}})

print("Pembaruan data selesai.")

Pembaruan data selesai.


D. Integrasi ke Spark

In [15]:
import numpy as np

# Konversi melalui Pandas DataFrame untuk menangani nilai null pada field yang tidak seragam
df_pandas = pd.DataFrame(data_dari_mongo)

# Ubah nilai NaN (float) bawaan Pandas menjadi None agar bisa dibaca sebagai Null oleh PySpark
df_pandas = df_pandas.replace({np.nan: None})

df_spark = spark.createDataFrame(df_pandas)
# Tampilkan skema DataFrame
df_spark.printSchema()

# Tampilkan rata-rata harga per kategori
df_spark.groupBy("kategori").avg("harga").show()

root
 |-- nama_menu: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- tingkat_kepahitan: double (nullable = true)
 |-- stok: double (nullable = true)
 |-- status: string (nullable = true)
 |-- rekomendasi_barista: boolean (nullable = true)
 |-- pedas_level: double (nullable = true)



+--------+------------------+
|kategori|        avg(harga)|
+--------+------------------+
|    Kopi|26333.333333333332|
| Makanan|23166.666666666668|
+--------+------------------+



----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 45144)
Traceback (most recent call last):
  File "/home/azizi/anaconda3/envs/bigdata/lib/python3.11/socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/home/azizi/anaconda3/envs/bigdata/lib/python3.11/socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "/home/azizi/anaconda3/envs/bigdata/lib/python3.11/socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/home/azizi/anaconda3/envs/bigdata/lib/python3.11/socketserver.py", line 755, in __init__
    self.handle()
  File "/home/azizi/anaconda3/envs/bigdata/lib/python3.11/site-packages/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/home/azizi/anaconda3/envs/bigdata/lib/python3.11/site-packages/pyspark/accumulators.py", line 267, in 

E. Kesimpulan

Tantangan utama saat mengonversi data MongoDB (NoSQL) ke Spark DataFrame adalah perbedaan sifat skema. MongoDB menerapkan flexible schema di mana dokumen dalam koleksi yang sama bisa memiliki field yang berbeda-beda (misalnya, menu Kopi memiliki tingkat_kepahitan, sedangkan menu Makanan memiliki pedas_level). Sebaliknya, Spark DataFrame mensyaratkan rigid schema atau format tabular yang seragam seperti tabel relasional. Saat data yang heterogen ini ditarik ke Spark, Spark akan memaksa pembentukan kolom untuk seluruh atribut yang ditemukan. Akibatnya, muncul banyak nilai kosong (null atau NaN) pada baris dokumen yang aslinya tidak memiliki field tersebut (seperti nilai tingkat_kepahitan pada menu Makanan menjadi null). Untuk mengatasinya, kita bisa meratakan data menggunakan perantara seperti Pandas DataFrame untuk memfasilitasi injeksi nilai null secara otomatis, lalu kita harus melakukan prapemrosesan tambahan di Spark seperti menggunakan fungsi fillna() untuk mengisi nilai default pada atribut yang kosong, atau hanya memilih atribut-atribut inti yang pasti dimiliki oleh semua dokumen jika data spesifik tidak diperlukan untuk analisis gabungan.